# Introduction

This notebook builds a leakage-safe dataset for image classification.

Problem:
- naive spatial splitting can place near-duplicate locations in both train and test

Solution:
- inspect and validate incoming NewData CSV
- apply graph-based connected-components fix (union-find) at 50m
- split by group (GroupShuffleSplit) to prevent spatial leakage
- export one final file: `ready_total_dataset.csv`

## Data Loading & Preparation

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import BallTree

ROOT = Path.cwd()
NEWDATA_DIR = ROOT / "NewData"
OLDDATA_DIR = ROOT / "OldData"
NEW_CSV = NEWDATA_DIR / "labeled_images.csv"
READY_CSV = ROOT / "ready_total_dataset.csv"

EARTH_RADIUS_KM = 6371.0
DIST_THRESHOLD_M = 50.0
MIN_SAMPLES = 2
TRAIN_SIZE = 0.70
RANDOM_STATE = 42

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

sns.set_theme(style="whitegrid")

setup_info = pd.DataFrame(
    {
        "item": ["Root", "New CSV", "Final output", "Distance threshold (m)", "Train ratio"],
        "value": [str(ROOT), str(NEW_CSV), str(READY_CSV), DIST_THRESHOLD_M, TRAIN_SIZE],
    }
)
display(setup_info)

### Haversine Distance Formula

The Haversine formula calculates the great-circle distance between two points on a sphere given their latitudes and longitudes:

$$d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$$

Where:
- $d$ = distance between the two points
- $R$ = radius of the Earth (6,371 km)
- $\phi_1, \phi_2$ = latitudes of the two points (in radians)
- $\lambda_1, \lambda_2$ = longitudes of the two points (in radians)
- $\Delta\phi = \phi_2 - \phi_1$ = difference in latitude
- $\Delta\lambda = \lambda_2 - \lambda_1$ = difference in longitude

This formula accounts for the spherical geometry of the Earth and is used in this notebook to identify nearby samples (within 50 meters) that should be kept in the same train/test split to prevent spatial leakage.


In [ ]:
def normalize_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip().lower()
    return re.sub(r"[^a-z0-9]+", "", s)

def list_image_paths(base_dir: Path):
    return [p for p in base_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]

def build_newdata_index(base_dir: Path):
    idx = {}
    duplicates = []
    for p in list_image_paths(base_dir):
        name = p.name
        if name in idx:
            duplicates.append((name, str(idx[name]), str(p)))
        else:
            idx[name] = p.resolve()
    return idx, duplicates

def infer_valid_status_values(df: pd.DataFrame):
    stats = (
        df.groupby("download_status", dropna=False)
        .agg(rows=("download_status", "size"), file_hits=("file_exists", "sum"))
        .reset_index()
    )
    stats["file_hit_rate"] = stats["file_hits"] / stats["rows"].clip(lower=1)
    keep = stats[stats["file_hit_rate"] >= 0.95]
    if keep.empty:
        keep = stats.sort_values(["file_hit_rate", "rows"], ascending=[False, False]).head(1)
    return keep["download_status"].astype(str).tolist(), stats.sort_values("rows", ascending=False)

def infer_label_maps(df_new: pd.DataFrame):
    map_df = df_new[["primary_label_id", "primary_label_name"]].dropna().copy()
    map_df["primary_label_id"] = pd.to_numeric(map_df["primary_label_id"], errors="coerce").astype("Int64")
    map_df = map_df.dropna(subset=["primary_label_id"]).copy()
    map_df["primary_label_id"] = map_df["primary_label_id"].astype(int)

    id_to_name = (
        map_df.groupby("primary_label_id")["primary_label_name"]
        .agg(lambda x: x.value_counts().index[0])
        .to_dict()
    )
    name_to_id = {normalize_text(v): k for k, v in id_to_name.items()}

    undefined_candidates = [k for k, v in id_to_name.items() if "undefined" in normalize_text(v)]
    undefined_id = undefined_candidates[0] if undefined_candidates else min(id_to_name.keys())

    defined_ids = sorted([k for k in id_to_name if k != undefined_id])
    defined_base_id = min(defined_ids) if defined_ids else undefined_id + 1

    return id_to_name, name_to_id, undefined_id, defined_base_id

def build_old_df(old_dir: Path, id_to_name: dict, name_to_id: dict, undefined_id: int, defined_base_id: int):
    rows = []
    for p in list_image_paths(old_dir):
        rel = p.relative_to(old_dir)
        parts = rel.parts

        label_code = None
        if len(parts) >= 2 and parts[0].startswith("0"):
            label_code = undefined_id
        elif len(parts) >= 3 and parts[0].startswith("1"):
            class_dir = parts[1]
            m = re.match(r"^\s*(\d+)\s*(.*)$", class_dir)
            if m:
                local_idx = int(m.group(1))
                local_name = normalize_text(m.group(2).strip().replace("-", "_").replace(" ", "_"))
                if local_name in name_to_id:
                    label_code = name_to_id[local_name]
                else:
                    label_code = defined_base_id + local_idx

        if label_code is None:
            if any("undefined" in normalize_text(x) for x in parts):
                label_code = undefined_id
            else:
                label_code = np.nan

        rows.append({
            "label_code": label_code,
            "label_name": id_to_name.get(int(label_code), "Unknown") if pd.notna(label_code) else "Unknown",
            "image_path": str(p.resolve()),
            "split": "train",
        })

    out = pd.DataFrame(rows)
    out["label_code"] = pd.to_numeric(out["label_code"], errors="coerce").astype("Int64")
    return out

def union_find_components(n, edges):
    parent = np.arange(n)
    rank = np.zeros(n, dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    for a, b in edges:
        union(a, b)

    roots = np.array([find(i) for i in range(n)])
    _, comp_ids = np.unique(roots, return_inverse=True)
    return comp_ids

def build_merged_clusters(df_in: pd.DataFrame):
    df = df_in.copy().reset_index(drop=False).rename(columns={"index": "orig_index"})
    df = df.dropna(subset=["latitude", "longitude"]).copy()

    coords_rad = np.radians(df[["latitude", "longitude"]].to_numpy(dtype=float))
    radius_rad = (DIST_THRESHOLD_M / 1000.0) / EARTH_RADIUS_KM

    tree = BallTree(coords_rad, metric="haversine")
    nbrs = tree.query_radius(coords_rad, r=radius_rad, return_distance=False)

    edges = []
    for i, arr in enumerate(nbrs):
        for j in arr:
            if j > i:
                edges.append((i, int(j)))

    df["merged_cluster_id"] = union_find_components(len(df), edges).astype(int)
    return df, len(edges)

In [ ]:
if not NEW_CSV.exists():
    raise FileNotFoundError(f"Missing CSV: {NEW_CSV}")

df_raw = pd.read_csv(NEW_CSV)

required_cols = [
    "file_name", "download_status", "date_iso",
    "primary_label_id", "primary_label_name",
    "latitude", "longitude", "labeled_by_user_id"
]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

df_raw["date_iso"] = pd.to_datetime(df_raw["date_iso"], errors="coerce", utc=True)
df_raw["primary_label_id"] = pd.to_numeric(df_raw["primary_label_id"], errors="coerce").astype("Int64")
df_raw["latitude"] = pd.to_numeric(df_raw["latitude"], errors="coerce")
df_raw["longitude"] = pd.to_numeric(df_raw["longitude"], errors="coerce")

file_index, duplicate_names = build_newdata_index(NEWDATA_DIR)

df_raw["matched_path"] = df_raw["file_name"].map(lambda x: str(file_index.get(str(x), "")))
df_raw["file_exists"] = df_raw["matched_path"].str.len() > 0

valid_status_values, status_stats = infer_valid_status_values(df_raw)
valid_df = df_raw[df_raw["download_status"].astype(str).isin(valid_status_values) & df_raw["file_exists"]].copy()
valid_df["image_path"] = valid_df["matched_path"]

summary_df = pd.DataFrame(
    {
        "metric": ["rows_in_csv", "valid_rows_selected", "missing_files", "duplicate_basenames"],
        "value": [len(df_raw), len(valid_df), int((~df_raw["file_exists"]).sum()), len(duplicate_names)],
    }
)

display(summary_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.barplot(data=summary_df, x="metric", y="value", ax=axes[0], color="#4C78A8")
axes[0].set_yscale("log")
axes[0].set_title("Data Loading Summary (log scale)")
axes[0].set_xlabel("")
axes[0].set_ylabel("count (log)")
axes[0].tick_params(axis="x", rotation=25)
for p, v in zip(axes[0].patches, summary_df["value"]):
    axes[0].annotate(f"{int(v)}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)

status_plot = status_stats.copy()
status_plot["download_status"] = status_plot["download_status"].astype(str)
sns.barplot(data=status_plot, x="download_status", y="rows", ax=axes[1], color="#F58518")
axes[1].set_yscale("log")
axes[1].set_title("download_status Counts (log scale)")
axes[1].set_xlabel("")
axes[1].set_ylabel("rows (log)")
axes[1].tick_params(axis="x", rotation=20)
for p, v in zip(axes[1].patches, status_plot["rows"]):
    axes[1].annotate(f"{int(v)}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## Exploratory Data Analysis (EDA)

In [ ]:
label_counts = valid_df["primary_label_id"].value_counts().sort_index()
user_counts = valid_df["labeled_by_user_id"].astype(str).value_counts(dropna=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

label_df = label_counts.rename_axis("label_code").reset_index(name="count")
sns.barplot(data=label_df, x="label_code", y="count", ax=axes[0], color="#54A24B")
axes[0].set_title("Valid NewData: Label Distribution")
axes[0].set_xlabel("label_code")
axes[0].set_ylabel("count")

user_df = user_counts.rename_axis("labeled_by_user_id").reset_index(name="count")
sns.barplot(data=user_df, x="labeled_by_user_id", y="count", ax=axes[1], color="#E45756")
axes[1].set_title("Valid NewData: Top Labelers")
axes[1].set_xlabel("")
axes[1].set_ylabel("count")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## Spatial Clustering & Leakage Fix

Apply graph-based connected-components fix (union-find) at 50m threshold.

The merged components are used as groups for splitting to enforce spatial independence.

In [ ]:
# Apply graph-based merged clustering using union-find at 50m threshold
merged_df, edge_count = build_merged_clusters(valid_df)

cluster_summary = pd.DataFrame(
    {
        "metric": ["merge_edges_under_50m", "merged_clusters"],
        "value": [edge_count, merged_df["merged_cluster_id"].nunique()],
    }
)

display(cluster_summary)

plt.figure(figsize=(8.5, 4.5))
ax = sns.barplot(data=cluster_summary, x="metric", y="value", color="#72B7B2")
ax.set_yscale("log")
plt.title("Spatial Clustering Summary (log scale)")
plt.xlabel("")
plt.ylabel("count (log)")
plt.xticks(rotation=20)
for p, v in zip(ax.patches, cluster_summary["value"]):
    ax.annotate(f"{int(v)}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## Train/Test Split

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, train_size=TRAIN_SIZE, random_state=RANDOM_STATE)
idx_train, idx_test = next(splitter.split(merged_df, groups=merged_df["merged_cluster_id"]))

new_train = merged_df.iloc[idx_train].copy()
new_test = merged_df.iloc[idx_test].copy()
new_train["split"] = "train"
new_test["split"] = "test"
new_split_df = pd.concat([new_train, new_test], ignore_index=True)

# No group may appear in both train and test
overlap = set(new_train["merged_cluster_id"]).intersection(set(new_test["merged_cluster_id"]))
if overlap:
    raise RuntimeError(f"Group leakage detected: {len(overlap)} shared merged clusters.")

split_counts = pd.DataFrame(
    {
        "split": ["train", "test"],
        "count": [len(new_train), len(new_test)],
    }
)
display(split_counts)

plt.figure(figsize=(6, 4))
sns.barplot(data=split_counts, x="split", y="count", hue="split", palette=["#4C78A8", "#F58518"], legend=False)
plt.title("Train/Test Split Size")
plt.xlabel("")
plt.ylabel("samples")
plt.tight_layout()
plt.show()

## Final Dataset Construction

In [ ]:
id_to_name, name_to_id, undefined_id, defined_base_id = infer_label_maps(df_raw)

old_df = build_old_df(OLDDATA_DIR, id_to_name, name_to_id, undefined_id, defined_base_id)

new_ready = new_split_df[["primary_label_id", "primary_label_name", "image_path", "split"]].copy()
new_ready = new_ready.rename(columns={"primary_label_id": "label_code", "primary_label_name": "label_name"})
new_ready["label_code"] = pd.to_numeric(new_ready["label_code"], errors="coerce").astype("Int64")
new_ready["label_name"] = new_ready["label_code"].map(id_to_name).fillna(new_ready["label_name"]).astype(str)
new_ready["image_path"] = new_ready["image_path"].astype(str)

ready_df = pd.concat([old_df, new_ready], ignore_index=True)
ready_df = ready_df[["label_code", "label_name", "image_path", "split"]].copy()

# Strict quality checks
if ready_df.isna().any().any():
    na_counts = ready_df.isna().sum()
    raise RuntimeError(f"Missing values found:\n{na_counts}")

ready_df = ready_df.drop_duplicates(subset=["label_code", "label_name", "image_path", "split"]).copy()

path_exists = ready_df["image_path"].map(lambda p: Path(p).exists())
if not path_exists.all():
    missing_n = int((~path_exists).sum())
    raise RuntimeError(f"Invalid paths found in final dataset: {missing_n}")

ready_df.to_csv(READY_CSV, index=False)

build_summary = pd.DataFrame(
    {
        "artifact": ["ready_total_dataset.csv", "rows_written", "valid_paths", "duplicates_removed"],
        "value": [str(READY_CSV), len(ready_df), int(path_exists.sum()), "Yes"],
    }
)
display(build_summary)

## Validation Summary

In [ ]:
summary_counts = pd.DataFrame(
    {
        "metric": ["total_samples", "train_samples", "test_samples", "leakage_clusters_in_both_splits"],
        "value": [
            len(ready_df),
            int((ready_df["split"] == "train").sum()),
            int((ready_df["split"] == "test").sum()),
            int((new_split_df.groupby("merged_cluster_id")["split"].nunique() > 1).sum()),
        ],
    }
)

display(summary_counts)

plt.figure(figsize=(8, 4))
sns.barplot(data=summary_counts, x="metric", y="value", color="#B279A2")
plt.title("Final Validation Summary")
plt.xlabel("")
plt.ylabel("count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

label_split_df = (
    ready_df.groupby(["split", "label_code", "label_name"]) 
    .size()
    .rename("count")
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=label_split_df, x="label_name", y="count", hue="split")
plt.title("Label Distribution by Split")
plt.xlabel("")
plt.ylabel("count")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## Class Distribution by Split

This section shows class balance across train and test using both counts and percentages.

In [ ]:
# Use in-memory final dataframe when available, otherwise load from exported CSV.
if "ready_df" in globals() and isinstance(ready_df, pd.DataFrame):
    dist_df = ready_df.copy()
else:
    dist_df = pd.read_csv(READY_CSV)

count_table = (
    dist_df.groupby(["split", "label_code", "label_name"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["split", "label_code"])
)

split_totals = count_table.groupby("split")["count"].transform("sum")
count_table["pct_in_split"] = (count_table["count"] / split_totals * 100).round(2)

count_pivot = count_table.pivot(index=["label_code", "label_name"], columns="split", values="count").fillna(0).astype(int)
pct_pivot = count_table.pivot(index=["label_code", "label_name"], columns="split", values="pct_in_split").fillna(0.0)

# Heatmap for count distribution
plt.figure(figsize=(7, 4.5))
sns.heatmap(count_pivot, annot=True, fmt="d", cmap="Blues")
plt.title("Class Distribution Heatmap (Counts)")
plt.xlabel("split")
plt.ylabel("class")
plt.tight_layout()
plt.show()

# Heatmap for percentage distribution
plt.figure(figsize=(7, 4.5))
sns.heatmap(pct_pivot, annot=True, fmt=".2f", cmap="Greens")
plt.title("Class Distribution Heatmap (% within split)")
plt.xlabel("split")
plt.ylabel("class")
plt.tight_layout()
plt.show()

# Split totals bar chart
split_total_df = dist_df["split"].value_counts().rename_axis("split").reset_index(name="count")
plt.figure(figsize=(5, 4))
sns.barplot(data=split_total_df, x="split", y="count", palette=["#4C78A8", "#F58518"])
plt.title("Split Totals")
plt.xlabel("")
plt.ylabel("count")
plt.tight_layout()
plt.show()

display(count_pivot)
display(pct_pivot)

In [ ]:
# Publication-Quality Class Distribution Figure
# Load the final dataset and create a professional publication-ready plot

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the final dataset
ready_df = pd.read_csv(READY_CSV)

# Aggregate counts by label_name and split
class_dist = ready_df.groupby(['label_name', 'split']).size().unstack(fill_value=0)
class_dist = class_dist.sort_index()  # Sort by label_name alphabetically

# Get unique labels sorted by label_code
label_map = ready_df[['label_code', 'label_name']].drop_duplicates().sort_values('label_code')
class_order = label_map['label_name'].tolist()
class_dist = class_dist.reindex(class_order)

# Ensure consistent column order
if 'train' in class_dist.columns and 'test' in class_dist.columns:
    class_dist = class_dist[['train', 'test']]

# Create the figure
fig, ax = plt.subplots(figsize=(10, 6))

# Define bar positions
x = np.arange(len(class_dist))
width = 0.35

# Create bars
bars1 = ax.bar(x - width/2, class_dist['train'], width, label='Train', color='C0')
bars2 = ax.bar(x + width/2, class_dist['test'], width, label='Test', color='C1')

# Add count annotations above bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=11, fontweight='normal')

# Formatting
ax.set_xlabel('Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=14, fontweight='bold')
ax.set_title('Class Distribution of the Dataset (Train vs Test)', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(class_dist.index, rotation=30, ha='right', fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=12, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Compute and print percentage distribution
print("\n" + "="*60)
print("CLASS DISTRIBUTION SUMMARY")
print("="*60)

total = len(ready_df)
for class_name in class_order:
    count = len(ready_df[ready_df['label_name'] == class_name])
    pct = (count / total) * 100
    print(f"{class_name:20s} | Count: {count:5d} ({pct:5.2f}%)")

print("-"*60)
print(f"{'Total':20s} | Count: {total:5d} (100.00%)")
print("="*60)

# Print split distribution
print("\nSPLIT DISTRIBUTION")
print("="*60)
for split in ['train', 'test']:
    split_count = len(ready_df[ready_df['split'] == split])
    split_pct = (split_count / total) * 100
    print(f"{split.upper():20s} | Count: {split_count:5d} ({split_pct:5.2f}%)")
print("="*60)


In [ ]:
# Unified 2x2 publication dashboard (figures 1, 2, 5, 6 only)
# Uses in-memory ready_df when available, otherwise loads exported CSV.
if "ready_df" in globals() and isinstance(ready_df, pd.DataFrame):
    ready_vis = ready_df.copy()
else:
    ready_vis = pd.read_csv(READY_CSV)

# Consistent source color code across all plots
COLOR_OLD = "#4C78A8"
COLOR_NEW = "#F58518"

# Ensure expected columns exist
required_cols = {"label_code", "label_name", "image_path", "split"}
missing_cols = required_cols - set(ready_vis.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for visualization: {sorted(missing_cols)}")

# Infer source (old/new) from image path roots
new_root = str(NEWDATA_DIR.resolve()).lower()
old_root = str(OLDDATA_DIR.resolve()).lower()

def infer_source(path_value):
    p = str(path_value).lower()
    if p.startswith(new_root):
        return "new"
    if p.startswith(old_root):
        return "old"
    return "unknown"

ready_vis["source"] = ready_vis["image_path"].map(infer_source)

# Keep class order stable by label_code
class_map = (
    ready_vis[["label_code", "label_name"]]
    .drop_duplicates()
    .sort_values("label_code")
)
class_names = class_map["label_name"].astype(str).tolist()

new_test_df = ready_vis[(ready_vis["source"] == "new") & (ready_vis["split"] == "test")].copy()

# Build training distribution by source (old vs new)
train_by_source = (
    ready_vis[ready_vis["split"] == "train"]
    .groupby(["label_name", "source"])
    .size()
    .unstack(fill_value=0)
)
if "old" in train_by_source.columns and "new" in train_by_source.columns:
    train_by_source = train_by_source[["old", "new"]]
elif "old" in train_by_source.columns:
    train_by_source["new"] = 0
    train_by_source = train_by_source[["old", "new"]]
else:
    train_by_source["old"] = 0
    train_by_source = train_by_source[["old", "new"]]
train_by_source = train_by_source.reindex(class_names)


def plot_class_bar(ax, frame, title, color):
    counts = frame["label_name"].astype(str).value_counts().reindex(class_names, fill_value=0)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color=color)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=25)
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=9)


def plot_training_old_vs_new(ax):
    x = np.arange(len(train_by_source))
    width = 0.35
    bars_old = ax.bar(
        x - width / 2,
        train_by_source["old"],
        width,
        label="Old Data",
        color=COLOR_OLD,
        edgecolor="black",
        linewidth=0.8,
    )
    bars_new = ax.bar(
        x + width / 2,
        train_by_source["new"],
        width,
        label="New Data",
        color=COLOR_NEW,
        edgecolor="black",
        linewidth=0.8,
    )
    ax.set_title("Training Data: Old vs New by Class", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.set_xticks(x)
    ax.set_xticklabels(train_by_source.index, rotation=25, fontsize=9)
    ax.legend(fontsize=9, loc="upper right")
    ax.tick_params(axis="x", labelsize=9)

    for bars in [bars_old, bars_new]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                    f"{int(height)}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )


def plot_source_pie(ax, split_name):
    split_df = ready_vis[ready_vis["split"] == split_name]
    source_counts = split_df["source"].value_counts().reindex(["old", "new"], fill_value=0)
    positive = source_counts[source_counts > 0]

    if positive.empty:
        ax.text(0.5, 0.5, f"No {split_name} samples", ha="center", va="center", fontsize=11)
        ax.axis("off")
        return

    colors = [COLOR_OLD if s == "old" else COLOR_NEW for s in positive.index]
    ax.pie(
        positive.values,
        labels=[f"{s} ({int(v)})" for s, v in positive.items()],
        autopct="%1.1f%%",
        startangle=90,
        colors=colors,
        wedgeprops={"linewidth": 1, "edgecolor": "white"},
        textprops={"fontsize": 10},
    )
    ax.set_title(f"{split_name.capitalize()} Source Contribution", fontweight="bold")


fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Figure 1
plot_training_old_vs_new(axes[0, 0])

# Figure 2 (new-only, so use source-consistent NEW color)
plot_class_bar(axes[0, 1], new_test_df, "New Data Class Distribution (test)", COLOR_NEW)

# Figure 5
plot_source_pie(axes[1, 0], "train")

# Figure 6
plot_source_pie(axes[1, 1], "test")

fig.suptitle("Dataset Overview: Training Data Source and Test Distribution", fontsize=16, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])

plt.show()


In [ ]:
# Publication-Quality Source Totals Bar Chart
# Overall data source distribution (old vs new) for the complete dataset

source_totals = ready_vis["source"].value_counts().reindex(["old", "new"], fill_value=0)

fig, ax = plt.subplots(figsize=(10, 7))

# Create bars with consistent color scheme
bars = ax.bar(source_totals.index, source_totals.values, color=["#4C78A8", "#F58518"], edgecolor="black", linewidth=1.5)

# Add count annotations
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}', 
            ha='center', va='bottom', fontsize=13, fontweight='bold')

# Formatting for publication
ax.set_title("Overall Data Source Distribution", fontsize=16, fontweight="bold", pad=20)
ax.set_xlabel("Data Source", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Samples", fontsize=14, fontweight="bold")
ax.tick_params(axis='both', labelsize=12)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Print source distribution summary
print("\n" + "="*60)
print("SOURCE DISTRIBUTION SUMMARY")
print("="*60)
total_samples = source_totals.sum()
for source in ["old", "new"]:
    count = source_totals.get(source, 0)
    pct = (count / total_samples * 100) if total_samples > 0 else 0
    print(f"{source.upper():10s} | Count: {count:5d} ({pct:5.2f}%)")
print("-"*60)
print(f"{'TOTAL':10s} | Count: {total_samples:5d} (100.00%)")
print("="*60)


## Spatial Distance Verification

Visual proof of spatial split effectiveness: demonstrating that test samples are spatially separated from training samples by measuring actual distances.

In [ ]:
# Visual proof of spatial split (fast): 2 test samples vs 3 training samples each (same class)
# Layout per figure: 1 test image on top, 3 train images below, arrows with distance labels.

from PIL import Image
from matplotlib.patches import ConnectionPatch

# Ensure split dataframe exists in the current kernel
if "new_split_df" not in globals() or not isinstance(new_split_df, pd.DataFrame):
    raise RuntimeError("new_split_df not found. Run the split cell first.")

required = {"split", "primary_label_name", "image_path", "latitude", "longitude"}
missing = required - set(new_split_df.columns)
if missing:
    raise KeyError(f"Missing columns in new_split_df: {sorted(missing)}")

split_geo = new_split_df.dropna(subset=["latitude", "longitude"]).copy()
split_geo["primary_label_name"] = split_geo["primary_label_name"].astype(str)

train_geo = split_geo[split_geo["split"] == "train"].copy()
test_geo = split_geo[split_geo["split"] == "test"].copy()


def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * EARTH_RADIUS_KM * 1000.0 * np.arcsin(np.sqrt(a))


def pick_test_samples(df_test, preferred_classes, n_samples=2):
    chosen = []

    for cls in preferred_classes:
        cls_df = df_test[df_test["primary_label_name"] == cls]
        if not cls_df.empty:
            chosen.append(cls_df.sample(1, random_state=42).iloc[0])

    if len(chosen) < n_samples:
        used_classes = {r["primary_label_name"] for r in chosen}
        extra_pool = df_test[~df_test["primary_label_name"].isin(used_classes)]
        if not extra_pool.empty:
            extra = (
                extra_pool.groupby("primary_label_name", group_keys=False)
                .apply(lambda x: x.sample(1, random_state=42))
                .reset_index(drop=True)
            )
            for _, row in extra.iterrows():
                if len(chosen) < n_samples:
                    chosen.append(row)

    return chosen[:n_samples]


def get_closest_train_same_class(test_row, df_train, k=3):
    same_class = df_train[df_train["primary_label_name"] == test_row["primary_label_name"]].copy()
    if same_class.empty:
        return same_class

    same_class["distance_m"] = haversine_m(
        test_row["latitude"],
        test_row["longitude"],
        same_class["latitude"].to_numpy(),
        same_class["longitude"].to_numpy(),
    )
    return same_class.sort_values("distance_m").head(k)


def load_img(path_str):
    p = Path(path_str)
    if not p.exists():
        return None
    try:
        return np.array(Image.open(p).convert("RGB"))
    except Exception:
        return None


def plot_top_vs_bottom3(test_row, close_train_df, fig_idx):
    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1.0], hspace=0.35)

    ax_top = fig.add_subplot(gs[0, :])
    bottom_axes = [fig.add_subplot(gs[1, i]) for i in range(3)]

    # Top: test image
    test_img = load_img(test_row["image_path"])
    if test_img is not None:
        ax_top.imshow(test_img)
    else:
        ax_top.text(0.5, 0.5, "Image not found", ha="center", va="center", fontsize=12)
    ax_top.axis("off")
    ax_top.set_title(
        f"Test Image (Class: {test_row['primary_label_name']})",
        fontsize=14,
        fontweight="bold",
        pad=10,
    )

    # Bottom: 3 nearest train images
    for i, ax in enumerate(bottom_axes):
        if i < len(close_train_df):
            row = close_train_df.iloc[i]
            img = load_img(row["image_path"])
            if img is not None:
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, "Image not found", ha="center", va="center")
            ax.set_title(f"Train #{i+1}", fontsize=11, fontweight="bold")
        else:
            row = None
            ax.text(0.5, 0.5, "Not enough\ntrain samples", ha="center", va="center", fontsize=11)
        ax.axis("off")

        # Draw arrow from top image to bottom image and annotate distance on arrow
        if row is not None:
            con = ConnectionPatch(
                xyA=(0.5, 0.02),
                coordsA=ax_top.transAxes,
                xyB=(0.5, 0.98),
                coordsB=ax.transAxes,
                arrowstyle="->",
                mutation_scale=16,
                lw=2.0,
                color="#B22222",
                shrinkA=2,
                shrinkB=2,
            )
            fig.add_artist(con)

            # Put distance text near the middle of the arrow in figure coordinates
            p1 = fig.transFigure.inverted().transform(ax_top.transAxes.transform((0.5, 0.02)))
            p2 = fig.transFigure.inverted().transform(ax.transAxes.transform((0.5, 0.98)))
            mx, my = (p1[0] + p2[0]) / 2.0, (p1[1] + p2[1]) / 2.0
            fig.text(
                mx,
                my,
                f"{row['distance_m']:.1f} m",
                ha="center",
                va="center",
                fontsize=10,
                fontweight="bold",
                color="#B22222",
                bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "#B22222", "alpha": 0.85},
            )

    min_dist = close_train_df["distance_m"].min() if not close_train_df.empty else np.nan
    fig.suptitle(
        f"Distance Demo {fig_idx}: 1 Test vs 3 Train Samples (Class: {test_row['primary_label_name']}) | Min distance = {min_dist:.1f} m",
        fontsize=15,
        fontweight="bold",
        y=0.98,
    )

    plt.show()


preferred = ["One_Track_Partly", "Fully"]
selected_tests = pick_test_samples(test_geo, preferred_classes=preferred, n_samples=2)

if len(selected_tests) < 2:
    raise RuntimeError("Could not select 2 valid test samples with geo coordinates.")

for idx, test_sample in enumerate(selected_tests, start=1):
    nearest_train = get_closest_train_same_class(test_sample, train_geo, k=3)
    if nearest_train.empty:
        print(f"No train samples found for class {test_sample['primary_label_name']}. Skipping figure {idx}.")
        continue
    plot_top_vs_bottom3(test_sample, nearest_train, fig_idx=idx)

print("\nDone: Generated two separate 1-vs-3 distance demo plots with arrows and labels.")
